# Gold · Hechos de Pedidos

## Propósito

Construir la tabla de hechos de pedidos en la capa Gold, aplicando las reglas de negocio del reporte y conservando el historial de cambios de estado de cada pedido (SCD2).

## Flujo

1. Leer los pedidos desde la capa Silver y las dimensiones vigentes de Gold.
2. Aplicar las reglas de negocio y la validación cruzada del riesgo de entrega.
3. Relacionar cada pedido con su cliente, ubicación, modo de envío, fecha de pedido y fecha de envío.
4. En la primera carga, crear la tabla con todos los pedidos como versión vigente.
5. En las siguientes ejecuciones, detectar los pedidos que cambiaron y registrar una nueva versión.
6. Cerrar las versiones anteriores e insertar las nuevas.

## Reglas de negocio

* Los pedidos `CANCELED` se excluyen del hecho.
* Los pedidos `SUSPECTED_FRAUD` se incluyen y se marcan con `es_sospechoso_fraude`.
* Las entregas con estado `Shipping canceled` se marcan con `es_excluido_otd_leadtime` para no entrar en On-Time Delivery ni en Lead Time.
* El riesgo de entrega tardía del origen (`late_delivery_risk_original`) se valida contra un cálculo propio (`late_delivery_risk_calculado`).

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_ft_orders`

**Llave sustituta:** `orden_key`

**Grano:** un registro por pedido vigente

## Atributos que generan una nueva versión

* Estado del pedido y estado de entrega
* Riesgo de entrega (origen y calculado)
* Días de envío reales y programados
* Marcas de fraude y de exclusión

## Resultado esperado

La tabla de hechos de pedidos actualizada, base del resumen por pedido y de los indicadores del reporte. Las consultas de negocio filtran `es_vigente = true`.

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_PEDIDOS_FQN = f"{silver_catalog}.{silver_schema}.hd_pedidos"
GOLD_DIM_CUSTOMERS_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_customers"
GOLD_DIM_LOCATIONS_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_locations"
GOLD_DIM_FECHA_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_fecha"
GOLD_DIM_SHIPPING_MODE_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_shipping_mode"
GOLD_FACT_ORDERS_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_orders"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_PEDIDOS_FQN}")
print(f"✓ Destino: {GOLD_FACT_ORDERS_FQN}")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_fact_orders"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 3: Leer Silver + dimensiones
# ============================================================

from pyspark.sql import functions as F

df_pedidos = spark.table(SILVER_PEDIDOS_FQN)

df_dim_customers = (
    spark.table(GOLD_DIM_CUSTOMERS_FQN)
    .filter(F.col("es_vigente") == True)
    .select("customer_id", "cliente_key")
)

df_dim_locations = spark.table(GOLD_DIM_LOCATIONS_FQN).select(
    "market", "order_region", "order_country", "order_state",
    "order_city", "order_zipcode", "location_key"
)

df_dim_fecha = spark.table(GOLD_DIM_FECHA_FQN).select("fecha", "fecha_key")

df_dim_shipping_mode = spark.table(GOLD_DIM_SHIPPING_MODE_FQN).select(
    "shipping_mode", "shipping_mode_key"
)

print(f"✓ Pedidos en Silver: {df_pedidos.count()}")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 4: Reglas de negocio + validación cruzada
# ------------------------------------------------------------
# - Excluye CANCELED del hecho por completo
# - Marca (no excluye) SUSPECTED_FRAUD
# - Marca los que deben excluirse de OTD/Lead Time (Shipping canceled)
# - Valida late_delivery_risk contra un cálculo propio
# ============================================================

df_pedidos_reglas = (
    df_pedidos
    .filter(F.col("order_status") != "CANCELED")
    .withColumn(
        "es_sospechoso_fraude",
        F.col("order_status") == "SUSPECTED_FRAUD"
    )
    .withColumn(
        "es_excluido_otd_leadtime",
        F.col("delivery_status") == "Shipping canceled"
    )
    .withColumn(
        "late_delivery_risk_calculado",
        (F.col("days_for_shipping_real") > F.col("days_for_shipment_scheduled")).cast("int")
    )
    .withColumnRenamed("late_delivery_risk", "late_delivery_risk_original")
    .withColumn(
        "riesgo_validado",
        F.col("late_delivery_risk_original") == F.col("late_delivery_risk_calculado")
    )
)

print(f"✓ Pedidos tras excluir CANCELED: {df_pedidos_reglas.count()}")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 5: Join contra las dimensiones (snapshot actual)
# ============================================================

df_fact_orders_actual = (
    df_pedidos_reglas.alias("p")
    .join(df_dim_customers.alias("c"), F.col("p.order_customer_id") == F.col("c.customer_id"), "left")
    .join(
        df_dim_locations.alias("l"),
        (F.col("p.market") == F.col("l.market")) &
        (F.col("p.order_region") == F.col("l.order_region")) &
        (F.col("p.order_country") == F.col("l.order_country")) &
        (F.col("p.order_state") == F.col("l.order_state")) &
        (F.col("p.order_city") == F.col("l.order_city")) &
        (F.col("p.order_zipcode") == F.col("l.order_zipcode")),
        "left"
    )
    .join(df_dim_fecha.alias("fp"), F.col("p.order_date") == F.col("fp.fecha"), "left")
    .join(df_dim_fecha.alias("fe"), F.col("p.shipping_date") == F.col("fe.fecha"), "left")
    .join(df_dim_shipping_mode.alias("sm"), F.col("p.shipping_mode") == F.col("sm.shipping_mode"), "left")
    .select(
        F.col("p.order_id"),
        F.col("c.cliente_key"),
        F.col("l.location_key"),
        F.col("fp.fecha_key").alias("fecha_pedido_key"),
        F.col("fe.fecha_key").alias("fecha_envio_key"),
        F.col("sm.shipping_mode_key"),
        F.col("p.order_status"),
        F.col("p.delivery_status"),
        F.col("p.late_delivery_risk_original"),
        F.col("p.late_delivery_risk_calculado"),
        F.col("p.riesgo_validado"),
        F.col("p.days_for_shipping_real"),
        F.col("p.days_for_shipment_scheduled"),
        F.col("p.es_sospechoso_fraude"),
        F.col("p.es_excluido_otd_leadtime"),
        F.col("p.fecha_inicio_corte"),
        F.col("p.fecha_fin_corte"),
        F.col("p.fecha_carga"),
        F.col("p.id_ejecucion")
    )
)

nulos_cliente = df_fact_orders_actual.filter(F.col("cliente_key").isNull()).count()
nulos_location = df_fact_orders_actual.filter(F.col("location_key").isNull()).count()

print(f"✓ Filas en snapshot actual: {df_fact_orders_actual.count()}")
print(f"⚠️ Sin match en dim_customers: {nulos_cliente}")
print(f"⚠️ Sin match en dim_locations: {nulos_location}")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 6: Crear tabla si no existe (primera carga)
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(GOLD_FACT_ORDERS_FQN):

    df_inicial = (
        df_fact_orders_actual
        .withColumn("orden_key", F.monotonically_increasing_id())
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    (
        df_inicial.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_FACT_ORDERS_FQN)
    )

    print(f"✓ Tabla {GOLD_FACT_ORDERS_FQN} creada con {df_inicial.count()} pedidos (carga inicial)")
    CARGA_INICIAL = True
else:
    print(f"✓ Tabla {GOLD_FACT_ORDERS_FQN} ya existe — se procesa como SCD2")
    CARGA_INICIAL = False

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 7: Lógica SCD2 (solo si la tabla ya existía)
# ------------------------------------------------------------
# Detecta cambios en el estado del pedido (order_status,
# delivery_status) o en sus cálculos de riesgo/entrega — el
# tipo de cambio que sí puede ocurrir de una corrida a otra.
# ============================================================

if not CARGA_INICIAL:

    delta_fact_orders = DeltaTable.forName(spark, GOLD_FACT_ORDERS_FQN)

    df_vigente = (
        delta_fact_orders.toDF()
        .filter(F.col("es_vigente") == True)
    )

    COLUMNAS_COMPARADAS = [
        "order_status", "delivery_status",
        "late_delivery_risk_original", "late_delivery_risk_calculado",
        "days_for_shipping_real", "days_for_shipment_scheduled",
        "es_sospechoso_fraude", "es_excluido_otd_leadtime"
    ]

    condicion_cambio = " OR ".join([
        f"v.{c} <> n.{c}" for c in COLUMNAS_COMPARADAS
    ])

    df_comparado = (
        df_fact_orders_actual.alias("n")
        .join(df_vigente.alias("v"), on="order_id", how="left")
        .selectExpr(
            "n.*",
            "v.orden_key AS orden_key_existente",
            f"CASE WHEN v.order_id IS NULL THEN 'NUEVO' "
            f"WHEN {condicion_cambio} THEN 'CAMBIO' "
            f"ELSE 'SIN_CAMBIO' END AS tipo_evento"
        )
    )

    df_a_cerrar = df_comparado.filter(F.col("tipo_evento") == "CAMBIO")
    df_a_insertar = df_comparado.filter(F.col("tipo_evento").isin("NUEVO", "CAMBIO"))

    print(f"✓ Pedidos nuevos: {df_comparado.filter(F.col('tipo_evento') == 'NUEVO').count()}")
    print(f"✓ Pedidos con cambios (nueva versión): {df_a_cerrar.count()}")
    print(f"✓ Pedidos sin cambios: {df_comparado.filter(F.col('tipo_evento') == 'SIN_CAMBIO').count()}")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 8: Cerrar versiones viejas (solo los que cambiaron)
# ============================================================

if not CARGA_INICIAL and df_a_cerrar.count() > 0:

    ids_con_cambio = [row["order_id"] for row in df_a_cerrar.select("order_id").collect()]

    delta_fact_orders.update(
        condition=(F.col("order_id").isin(ids_con_cambio)) & (F.col("es_vigente") == True),
        set={
            "es_vigente": F.lit(False),
            "fecha_fin_vigencia": F.current_date()
        }
    )

    print(f"✓ {len(ids_con_cambio)} versiones anteriores cerradas")
elif not CARGA_INICIAL:
    print("✓ No hay versiones que cerrar")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 9: Insertar nuevas versiones (nuevos + cambiados)
# ============================================================

if not CARGA_INICIAL and df_a_insertar.count() > 0:

    max_key = spark.sql(f"SELECT MAX(orden_key) AS m FROM {GOLD_FACT_ORDERS_FQN}").collect()[0]["m"] or 0

    df_nuevas_versiones = (
        df_a_insertar
        .drop("orden_key_existente", "tipo_evento")
        .withColumn("orden_key", max_key + F.monotonically_increasing_id() + 1)
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    df_nuevas_versiones.write.format("delta").mode("append").saveAsTable(GOLD_FACT_ORDERS_FQN)

    print(f"✓ {df_nuevas_versiones.count()} nuevas versiones insertadas")
elif not CARGA_INICIAL:
    print("✓ No hay nuevas versiones que insertar")

In [0]:
# ============================================================
# GLD_FACT_ORDERS — Celda 10: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")